# RSA model for a marked hope-wh utterance

This notebook is a small exploratory model. I use it to ask how listener assumptions about speaker background might change the interpretation of a marked sentence such as `I hope what will happen`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.append(str(Path("..").resolve()))
from src.rsa_model import RSAModel

## Messages and states

The model uses two possible speaker states and three possible messages. The target message is treated as marked, but still compatible with a positive-desire state in this first pass.

In [ ]:
objects = ["desire_positive_S", "uncertain_about_S"]
messages = ["hope_that_S_good", "wonder_what_S", "hope_what_S"]
truth_table = [[1, 0], [0, 1], [1, 0]]

def make_model(costs, prior_m):
    return RSAModel(objects, messages, truth_table, 3.0, [0.5, 0.5], prior_m, lambda: costs)

In [ ]:
native_model = make_model(costs=(0, 0, 2.0), prior_m=(0.49, 0.49, 0.02))
l2_model = make_model(costs=(0, 0, 0.5), prior_m=(0.4, 0.4, 0.2))

print("Native L1")
print(native_model.l1())
print("\nL2 L1")
print(l2_model.l1())

In [ ]:
def plot_l1(model, title, output_path):
    l1 = model.l1()
    x = range(len(messages))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([i - 0.18 for i in x], [row[0] for row in l1], width=0.35, label=objects[0])
    ax.bar([i + 0.18 for i in x], [row[1] for row in l1], width=0.35, label=objects[1])
    ax.set_xticks(list(x))
    ax.set_xticklabels(messages, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("L1 probability")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)

Path("../figures").mkdir(exist_ok=True)
plot_l1(native_model, "Native-speaker assumption", "../figures/native_model.png")
plot_l1(l2_model, "L2-speaker assumption", "../figures/l2_model.png")

## Cost sensitivity

This sweep asks how the speaker model changes as the marked `hope_what_S` message becomes more costly.

In [ ]:
cost_values = [i / 10 for i in range(31)]
marked_use = []
for cost in cost_values:
    model = make_model(costs=(0, 0, cost), prior_m=(0.4, 0.4, 0.2))
    marked_use.append(model.s1()[objects.index("desire_positive_S")][messages.index("hope_what_S")])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cost_values, marked_use, marker="o", markersize=3)
ax.set_xlabel("Cost assigned to hope_what_S")
ax.set_ylabel("S1(hope_what_S | desire_positive_S)")
ax.set_title("Cost sensitivity for the marked message")
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig("../figures/cost_sweep.png", dpi=150)

## Current interpretation

This is still a toy model. The interesting part is not the exact numeric output, but the way costs and priors make explicit a listener's assumptions about speaker background. A next step would be to replace hand-set values with judgment data.